In [ ]:
# Configuration
import os
from pyspark.sql import SparkSession, functions as F

IS_DATABRICKS = 'DATABRICKS_RUNTIME_VERSION' in os.environ

# Database connection
DB_HOST = os.environ.get('POSTGRES_HOST', 'localhost')
DB_PORT = os.environ.get('POSTGRES_PORT', '5432')
DB_NAME = os.environ.get('POSTGRES_DB', 'flex_finops')
DB_USER = os.environ.get('POSTGRES_USER', 'flex_admin')
DB_PASSWORD = os.environ.get('POSTGRES_PASSWORD', 'flex_local_dev_2026')

JDBC_URL = f'jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}?stringtype=unspecified'
JDBC_PROPS = {
    'user': DB_USER,
    'password': DB_PASSWORD,
    'driver': 'org.postgresql.Driver',
    'currentSchema': 'flex',
    'stringtype': 'unspecified',  # Allows PostgreSQL to auto-cast strings to UUID
}

YEAR = os.environ.get('YEAR', '2026')
MONTH = os.environ.get('MONTH', '05')

if not IS_DATABRICKS:
    spark = (SparkSession.builder
        .master('local[*]')
        .appName('Flex-CUR-Load')
        .config('spark.jars.packages', 'org.postgresql:postgresql:42.7.1')
        .config('spark.driver.memory', '4g')
        .getOrCreate())
else:
    spark = SparkSession.builder.getOrCreate()

print(f'JDBC URL: {JDBC_URL}')
print(f'Loading period: {YEAR}-{MONTH}')

In [ ]:
# Read transformed data from local parquet (output of Notebook 2)
import pandas as pd

data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data', 'transformed', f'{YEAR}_{MONTH}')

kpi_pdf = pd.read_parquet(f'{data_dir}/kpis.parquet')
usage_pdf = pd.read_parquet(f'{data_dir}/cloud_usage.parquet')
chargeback_pdf = pd.read_parquet(f'{data_dir}/chargeback.parquet')
savings_pdf = pd.read_parquet(f'{data_dir}/savings.parquet')

print(f'KPIs: {len(kpi_pdf)} rows')
print(f'Cloud Usage: {len(usage_pdf)} rows')
print(f'Chargeback: {len(chargeback_pdf)} rows')
print(f'Savings: {len(savings_pdf)} rows')

In [ ]:
# ===========================
# LOAD 1: KPIs
# ===========================

kpi_spark = spark.createDataFrame(kpi_pdf)

# Add default values for columns not yet computed
kpi_load = (kpi_spark
    .withColumn('utilization_pct', F.lit(78.0))  # Placeholder until resource data enriches
    .withColumn('open_anomalies', F.lit(0))  # Will be updated by anomaly notebook
    .withColumn('pending_approvals', F.lit(0))
    .withColumn('monthly_savings_identified', F.lit(0.0))
    .withColumn('monthly_savings_realized', F.lit(0.0))
    .withColumn('computed_at', F.current_timestamp())
    .withColumnRenamed('total_spend', 'total_spend')
)

# Write to PostgreSQL
(kpi_load
    .select('business_unit_id', 'period', 'total_spend', 'spend_change_pct',
            'utilization_pct', 'active_resources', 'open_anomalies',
            'pending_approvals', 'monthly_savings_identified', 'monthly_savings_realized', 'computed_at')
    .write
    .mode('append')  # Use append; upsert handled by DB constraint
    .jdbc(JDBC_URL, 'flex.kpis', properties=JDBC_PROPS)
)

print('✅ KPIs loaded to flex.kpis')

In [ ]:
# ===========================
# LOAD 2: Cloud Usage Time-Series (TimescaleDB hypertable)
# ===========================

# Unpivot from wide (compute/storage/network/database) to long format for cloud_usage table
usage_spark = spark.createDataFrame(usage_pdf)

# The cloud_usage table expects: time, business_unit_id, service, region, usage_amount, blended_cost, unblended_cost
# We'll write the pivoted daily totals as aggregated rows
categories = ['compute', 'storage', 'network', 'database']

usage_rows = []
for _, row in usage_pdf.iterrows():
    for cat in categories:
        cost_val = row.get(cat, 0) or 0
        if cost_val > 0:
            usage_rows.append({
                'time': row['time'],
                'business_unit_id': row['business_unit_id'],
                'service': cat,
                'region': 'us-east-1',  # Aggregated
                'usage_amount': cost_val,
                'blended_cost': cost_val * 0.98,
                'unblended_cost': cost_val,
            })

usage_load_pdf = pd.DataFrame(usage_rows)
usage_load_spark = spark.createDataFrame(usage_load_pdf)

(usage_load_spark
    .write
    .mode('append')
    .jdbc(JDBC_URL, 'flex.cloud_usage', properties=JDBC_PROPS)
)

print(f'✅ Cloud usage loaded: {len(usage_rows)} rows to flex.cloud_usage (hypertable)')

In [ ]:
# ===========================
# LOAD 3: Chargeback
# ===========================

chargeback_spark = spark.createDataFrame(chargeback_pdf)

chargeback_load = (chargeback_spark
    .withColumn('initiative', F.lit(''))  # To be enriched
    .withColumn('budget', F.col('monthly_spend') * 1.1)  # 10% buffer as placeholder
    .withColumn('forecast', F.col('monthly_spend') * 1.05)
    .withColumn('headcount', F.lit(0))  # Enriched from workforce
    .withColumn('tag_compliance_pct', F.lit(85.0))  # From tag compliance transform
    .withColumn('trend', F.lit('stable'))
)

(chargeback_load
    .select('business_unit_id', 'team', 'cost_center', 'initiative', 'owner',
            'monthly_spend', 'budget', 'forecast', 'headcount',
            'cost_per_engineer', 'tag_compliance_pct', 'trend', 'period')
    .write
    .mode('append')
    .jdbc(JDBC_URL, 'flex.chargeback', properties=JDBC_PROPS)
)

print(f'✅ Chargeback loaded: {chargeback_pdf.shape[0]} rows to flex.chargeback')

In [ ]:
# ===========================
# LOAD 4: Savings Opportunities
# ===========================

if len(savings_pdf) > 0:
    savings_spark = spark.createDataFrame(savings_pdf)

    savings_load = (savings_spark
        .withColumn('title', F.concat(
            F.lit('Idle '), F.col('service'), F.lit(' resource: '), F.col('lineItem/ResourceId')
        ))
        .withColumn('action', F.lit('Consider terminating or rightsizing'))
        .withColumn('stage', F.lit('identified'))
        .withColumn('owner', F.lit(''))
    )

    (savings_load
        .select('business_unit_id', 'title', 'category', 'monthly_savings',
                'effort', 'confidence', 'action', 'stage', 'owner')
        .write
        .mode('append')
        .jdbc(JDBC_URL, 'flex.savings', properties=JDBC_PROPS)
    )

    print(f'✅ Savings loaded: {len(savings_pdf)} rows to flex.savings')
else:
    print('ℹ️ No savings opportunities to load')

In [ ]:
# ===========================
# LOAD 5: Audit Log Entry
# ===========================
import hashlib, json
from datetime import datetime

# Record ETL run in audit log
audit_entry = {
    'business_unit_id': 'a1b2c3d4-0001-4000-8000-000000000001',  # System entry
    'action': 'etl.load.completed',
    'entity_type': 'etl_pipeline',
    'entity_id': f'cur_load_{YEAR}_{MONTH}',
    'payload': json.dumps({
        'kpi_rows': len(kpi_pdf),
        'usage_rows': len(usage_pdf),
        'chargeback_rows': len(chargeback_pdf),
        'savings_rows': len(savings_pdf),
        'period': f'{YEAR}-{MONTH}',
        'completed_at': datetime.now().isoformat(),
    }),
    'prev_hash': '0' * 64,
    'hash': hashlib.sha256(f'etl_load_{YEAR}_{MONTH}_{datetime.now().isoformat()}'.encode()).hexdigest(),
}

audit_df = spark.createDataFrame([audit_entry])
audit_df.write.mode('append').jdbc(JDBC_URL, 'flex.audit_log', properties=JDBC_PROPS)

print('✅ Audit log entry recorded')
print(f'\n🎉 ETL Load complete for {YEAR}-{MONTH}!')
print(f'   Tables populated: flex.kpis, flex.cloud_usage, flex.chargeback, flex.savings, flex.audit_log')